# S05 · Canonicalizing Atom-Mapped Reactions and Rules: Equivalence, Symmetry, Determinism

<div class="alert alert-block alert-info">
<b>Welcome to SynEdu.</b><br>
This talktorial is part of <b>SynEdu</b>, a lightweight, research-oriented teaching series built around the
<b>Syn</b> ecosystem and <b>RDKit</b> for practical, reproducible reaction modeling.
</div>

<div class="alert alert-block alert-success">
<b>What you will gain.</b><br>
Atom-mapped reactions are <b>not unique</b>: the same chemistry may admit many valid maps and many map-numberings.
In this notebook we turn mapped reactions into <b>deterministic, comparable objects</b> by canonicalizing (i) the <b>reaction string</b>
(order of components) and (ii) the <b>atom-map numbering</b>. We then show how canonicalization stabilizes
<b>reaction-center clustering</b> and <b>DPO rule extraction</b> downstream.
</div>

## Aim of this talktorial

## Learning outcomes
By the end of this notebook, you should be able to:
- Explain why atom maps are non-unique (symmetry) and why map numbers should be canonicalized.
- Implement a deterministic atom-map reindexing \(\pi: \mathbb{N} \to \mathbb{N}\) based on map-invariant structural ranks.
- Canonicalize a mapped reaction SMILES (molecule order + map ids) without changing chemistry.
- Quantify how canonicalization affects the number of unique centers/rules (hash + isomorphism).

## Roadmap
- 0. Setup & data (mapped reactions from `./data/smart.json.gz`)
- 1. Theory: equivalence of maps and why “canonical” matters
- 2. Canonicalizing mapped reaction SMILES
- 3. From canonical maps to canonical DPO rules
- 4. Mini-study: duplicates before/after
- 5. Exercises

<div class="alert alert-block alert-warning">
<b>Note.</b> This notebook assumes your dataset already contains <b>atom-mapped</b> reaction SMILES
(e.g., from RXNMapper). If the file is not present, the code will raise a clear error.
</div>


In [ ]:

from __future__ import annotations

import json
import gzip
from pathlib import Path
from typing import Any, Dict, Hashable, Iterable, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
import networkx as nx
import networkx.algorithms.isomorphism as iso

from rdkit import Chem
from rdkit.Chem import AllChem

import matplotlib.pyplot as plt

def load_json_auto(path: Union[str, Path]) -> Any:
    """
    Load JSON from .json or .json.gz.

    Auto-detects gzip by magic bytes, not filename.
    Works even if a file is misnamed *.json.gz but is plain JSON.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")

    with open(path, "rb") as fb:
        magic = fb.read(2)

    is_gzip = magic == b"\x1f\x8b"
    if is_gzip:
        with gzip.open(path, "rt", encoding="utf-8") as f:
            return json.load(f)
    with open(path, "rt", encoding="utf-8") as f:
        return json.load(f)

print("RDKit:", getattr(Chem, "__version__", "unknown"))
print("NetworkX:", nx.__version__)


In [ ]:

def to_records(obj: Any) -> List[Dict[str, Any]]:
    """
    Normalize a JSON object into a list of reaction records (dicts).

    Supports:
    - list[dict]
    - {"data": list[dict]} and common variants
    - dict[str, dict] (id -> record)
    """
    if isinstance(obj, list):
        return [r for r in obj if isinstance(r, dict)]
    if isinstance(obj, dict):
        for k in ("data", "reactions", "records", "items", "rxns"):
            v = obj.get(k)
            if isinstance(v, list) and v and isinstance(v[0], dict):
                return v
        # id -> record
        if obj and all(isinstance(v, dict) for v in obj.values()):
            return list(obj.values())
    raise TypeError(f"Unsupported data type: {type(obj)}")

def looks_like_mapped_rxn_smiles(s: str) -> bool:
    if not isinstance(s, str):
        return False
    if ">" not in s:
        return False
    # atom-map numbers typically appear as ":<int>]"
    return (":" in s) and ("]" in s) and any(tok in s for tok in (">>", ">"))

def guess_mapped_field(records: List[Dict[str, Any]]) -> str:
    """
    Heuristic: choose the first key whose value looks like an atom-mapped reaction SMILES.
    """
    if not records:
        raise ValueError("Empty records")
    keys = set().union(*(r.keys() for r in records[:50]))
    candidates: List[Tuple[int, str]] = []
    for k in keys:
        score = 0
        for r in records[:50]:
            v = r.get(k)
            if isinstance(v, str) and looks_like_mapped_rxn_smiles(v):
                score += 1
        if score:
            candidates.append((score, k))
    if not candidates:
        raise KeyError("Could not find a mapped reaction SMILES field automatically.")
    candidates.sort(reverse=True)
    return candidates[0][1]

def split_rxn_smiles(rxn: str) -> Tuple[List[str], List[str], List[str]]:
    """
    Split a reaction SMILES into (reactants, agents, products) lists.

    Accepts:
    - reactants>>products
    - reactants>agents>products
    """
    if ">>" in rxn:
        left, right = rxn.split(">>")
        return [s for s in left.split(".") if s], [], [s for s in right.split(".") if s]
    parts = rxn.split(">")
    if len(parts) == 3:
        left, mid, right = parts
        return [s for s in left.split(".") if s], [s for s in mid.split(".") if s], [s for s in right.split(".") if s]
    raise ValueError(f"Unrecognized reaction SMILES format: {rxn[:80]}...")


## 0. Setup & data

We load a JSON (optionally gzipped) file that contains **atom-mapped reaction SMILES**.
Throughout SynEdu we will treat the mapped reaction string as *raw evidence* from which we derive
graph objects (ITS, centers, rules).

We will do two things immediately:

1. Normalize the JSON into a list of records (`records`).
2. Guess which field contains the mapped reaction SMILES (`mapped_key`), since datasets vary.

> In many datasets the mapped reaction may appear under keys like `mapped_rxn`, `rxn_smiles_mapped`, `rxn`, etc.


In [ ]:

# --- Load the mapped reaction dataset (ready-to-use atom maps) ---
data = load_json_auto("./data/smart.json.gz")
print("Top-level type:", type(data))
print("len(data):", len(data))

# For quick inspection (like the snippet you asked for):
data


In [ ]:

records = to_records(data)
print("Number of reaction records:", len(records))

mapped_key = guess_mapped_field(records)
print("Guessed mapped field:", mapped_key)

# Build a DataFrame (keep all columns, but ensure "mapped_rxn" exists)
df = pd.DataFrame(records).copy()
df["mapped_rxn"] = df[mapped_key].astype(str)

display(df[["mapped_rxn"]].head(5))


## 1. Theory (formal, but chemist-friendly)

### 1.1 Atom maps as witnesses, not identities

For a reaction \(r\), let \(\mathcal{M}(r)\) denote the set of **valid atom mappings**.
A mapping \(m\in\mathcal{M}(r)\) is best viewed as a **witness** that reactant atoms correspond to product atoms.

In practice, \(|\mathcal{M}(r)|\) is often \(>1\) because of **molecular symmetry**:
if a molecule has automorphisms (e.g., benzene), then multiple atom correspondences are equally valid.

### 1.2 Two sources of non-uniqueness

Even if you fix the chemistry, two mapped reaction strings may differ by:

1. **Component order** (which molecule appears first on each side).
2. **Map-number permutation** (renaming atom map ids).

Formally, a **map renaming** is a bijection
\[
\pi:\{1,\dots,n\}\to\{1,\dots,n\},
\]
and it induces an equivalent mapped reaction by replacing every \([*:i]\) with \([*:\pi(i)]\).

### 1.3 Canonicalization as a quotient

We define an equivalence relation on mapped reactions:
\[
\rho_1 \sim \rho_2
\quad\Longleftrightarrow\quad
\rho_2 = \pi(\rho_1) \text{ for some map renaming }\pi \text{ (and possibly permuted components).}
\]

A **canonicalization** procedure chooses a representative
\[
\operatorname{can}(\rho)\in[\rho]
\]
so that equivalent inputs yield the same output.

**Chemistry meaning:** we do not change the reaction—only its *encoding*.

### 1.4 Why this matters downstream

If you extract a rule (or a reaction center) from a mapped reaction, map-number permutations
can create many syntactically different but chemically identical rules.

Canonicalization makes:
- **rule libraries reproducible** (stable hashing and clustering),
- **statistics meaningful** (counts of “unique rules” stop drifting),
- **benchmarking fair** (different mappers can be compared under the same quotient).


## 2. Canonicalizing mapped reaction SMILES

We canonicalize a mapped reaction in two stages:

1. **Canonicalize molecule order** on each side by sorting molecules by their **unmapped canonical SMILES**.
2. **Canonicalize map ids** by assigning new ids \(1,2,\dots\) in a deterministic order derived from
   **map-invariant RDKit canonical atom ranks** on each molecule.

### 2.1 Deterministic reindexing rule (practical)

For every atom-map id \(i\), we build a key

\[
\kappa(i) = \big(\text{side},\ \text{mol\_key},\ \text{rank},\ \text{symbol},\ \text{charge},\ \text{aromatic}\big),
\]

where:
- `side = 0` if the mapped atom occurs on the **reactant** side, else `1` (product-only atoms, if any);
- `mol_key` is the **unmapped canonical SMILES** of the parent molecule;
- `rank` is the RDKit **canonical rank** of the atom in that unmapped molecule.

Then we sort map ids by \(\kappa(i)\) and assign new ids \(1,2,\dots\).

This does **not** solve symmetry in a philosophically perfect way (symmetry cannot always be broken uniquely),
but RDKit's canonicalization gives a consistent tie-breaker that is independent of the original atom numbering.


In [ ]:

def clear_atom_maps(m: Chem.Mol) -> Chem.Mol:
    mm = Chem.Mol(m)
    for a in mm.GetAtoms():
        a.SetAtomMapNum(0)
    return mm

def mol_key_unmapped(m: Chem.Mol) -> str:
    """Canonical SMILES without atom maps, used to sort molecules deterministically."""
    return Chem.MolToSmiles(clear_atom_maps(m), canonical=True)

def canonical_ranks_unmapped(m: Chem.Mol) -> List[int]:
    """
    RDKit canonical ranks computed on the unmapped molecule.
    Returned list aligns with atom indices of the unmapped copy.
    """
    mm = clear_atom_maps(m)
    # includeChirality=True keeps stereochemistry consistent in ranking
    return list(Chem.CanonicalRankAtoms(mm, includeChirality=True))

def mapped_atoms_in_mol(m: Chem.Mol) -> List[Tuple[int, int]]:
    """Return (atom_idx, map_id) pairs for mapped atoms (map_id > 0)."""
    out: List[Tuple[int, int]] = []
    for a in m.GetAtoms():
        mid = int(a.GetAtomMapNum() or 0)
        if mid > 0:
            out.append((a.GetIdx(), mid))
    return out

def canonical_map_renaming(mapped_rxn: str) -> Dict[int, int]:
    """
    Compute a deterministic renaming old_map_id -> new_map_id for a mapped reaction SMILES.

    Key idea:
    - map ids observed on reactants are ordered first
    - within each side: molecules sorted by unmapped canonical SMILES
    - within each molecule: atoms ordered by RDKit canonical rank on the unmapped molecule
    """
    r_smis, _agents, p_smis = split_rxn_smiles(mapped_rxn)
    r_mols = [Chem.MolFromSmiles(s) for s in r_smis]
    p_mols = [Chem.MolFromSmiles(s) for s in p_smis]
    if any(m is None for m in r_mols + p_mols):
        raise ValueError("Failed to parse one or more molecules in reaction SMILES.")

    def collect_side(mols: List[Chem.Mol], side: int) -> List[Tuple[Tuple, int]]:
        items: List[Tuple[Tuple, int]] = []
        # deterministic molecule ordering
        mols_sorted = sorted(mols, key=mol_key_unmapped)
        for m in mols_sorted:
            key_m = mol_key_unmapped(m)
            ranks = canonical_ranks_unmapped(m)
            for aidx, mid in mapped_atoms_in_mol(m):
                a = m.GetAtomWithIdx(aidx)
                k = (
                    side,
                    key_m,
                    int(ranks[aidx]),
                    a.GetSymbol(),
                    int(a.GetFormalCharge()),
                    bool(a.GetIsAromatic()),
                )
                items.append((k, mid))
        return items

    seen: Dict[int, Tuple] = {}

    # Reactant side first (side=0)
    for k, mid in collect_side(r_mols, side=0):
        seen.setdefault(mid, k)

    # Then product-only map ids (side=1)
    for k, mid in collect_side(p_mols, side=1):
        seen.setdefault(mid, k)

    # Sort by the deterministic keys; assign new ids 1..n
    ordered = sorted([(k, mid) for mid, k in seen.items()])
    renaming: Dict[int, int] = {mid: i + 1 for i, (_k, mid) in enumerate(ordered)}
    return renaming

def apply_map_renaming_to_rxn(mapped_rxn: str, renaming: Dict[int, int]) -> str:
    """Apply old->new map renaming and return a canonicalized mapped reaction SMILES."""
    r_smis, agents, p_smis = split_rxn_smiles(mapped_rxn)
    r_mols = [Chem.MolFromSmiles(s) for s in r_smis]
    p_mols = [Chem.MolFromSmiles(s) for s in p_smis]
    a_mols = [Chem.MolFromSmiles(s) for s in agents] if agents else []

    for m in r_mols + p_mols + a_mols:
        if m is None:
            raise ValueError("Bad SMILES while applying renaming.")
        for a in m.GetAtoms():
            mid = int(a.GetAtomMapNum() or 0)
            if mid > 0 and mid in renaming:
                a.SetAtomMapNum(int(renaming[mid]))

    # deterministic molecule ordering by unmapped canonical SMILES
    def mol_to_mapped_smiles(m: Chem.Mol) -> str:
        return Chem.MolToSmiles(m, canonical=True)

    r_out = ".".join(mol_to_mapped_smiles(m) for m in sorted(r_mols, key=mol_key_unmapped))
    p_out = ".".join(mol_to_mapped_smiles(m) for m in sorted(p_mols, key=mol_key_unmapped))
    if agents:
        a_out = ".".join(mol_to_mapped_smiles(m) for m in sorted(a_mols, key=mol_key_unmapped))
        return f"{r_out}>{a_out}>{p_out}"
    return f"{r_out}>>{p_out}"

def canonicalize_mapped_rxn(mapped_rxn: str) -> str:
    ren = canonical_map_renaming(mapped_rxn)
    return apply_map_renaming_to_rxn(mapped_rxn, ren)


In [ ]:

import random

def permute_map_ids(mapped_rxn: str, seed: int = 0) -> str:
    """Randomly permute map ids (1..n) to simulate a different but equivalent numbering."""
    rng = random.Random(seed)
    r_smis, agents, p_smis = split_rxn_smiles(mapped_rxn)
    mols = [Chem.MolFromSmiles(s) for s in (r_smis + agents + p_smis)]
    mids = sorted({int(a.GetAtomMapNum()) for m in mols if m is not None for a in m.GetAtoms() if int(a.GetAtomMapNum() or 0) > 0})
    if not mids:
        return mapped_rxn
    perm = mids[:]
    rng.shuffle(perm)
    ren = {old: new for old, new in zip(mids, perm)}

    # apply renaming
    def apply_to_smiles_list(smis: List[str]) -> List[str]:
        out=[]
        for sm in smis:
            m = Chem.MolFromSmiles(sm)
            for a in m.GetAtoms():
                mid = int(a.GetAtomMapNum() or 0)
                if mid > 0:
                    a.SetAtomMapNum(int(ren[mid]))
            out.append(Chem.MolToSmiles(m, canonical=False))
        return out

    r2 = ".".join(apply_to_smiles_list(r_smis))
    p2 = ".".join(apply_to_smiles_list(p_smis))
    if agents:
        a2 = ".".join(apply_to_smiles_list(agents))
        return f"{r2}>{a2}>{p2}"
    return f"{r2}>>{p2}"

# --- Demonstration on a single reaction ---
ex = df["mapped_rxn"].dropna().iloc[0]
ex_perm = permute_map_ids(ex, seed=42)

ex_can = canonicalize_mapped_rxn(ex)
ex_perm_can = canonicalize_mapped_rxn(ex_perm)

print("Original mapped rxn:")
print(ex)
print("\nPermuted map ids:")
print(ex_perm)
print("\nCanonical(original):")
print(ex_can)
print("\nCanonical(permuted):")
print(ex_perm_can)

print("\nCanonical strings identical?", ex_can == ex_perm_can)


## 3. From canonical maps to canonical DPO rules

Once a mapped reaction is canonicalized, every downstream graph we build becomes more stable:

- ITS graphs use **map ids as node identifiers**.
- Reaction centers are extracted as subgraphs of the ITS.
- DPO rules \(L \leftarrow K \rightarrow R\) reuse those ids.

So if we canonicalize map ids up front, then:
- identical chemistry yields identical ITS node ids,
- “unique rule counts” stop depending on incidental numbering.

Below we reuse the same ITS + center + rule extraction ideas as S04, but apply them to:
- raw mapped reactions, and
- canonicalized mapped reactions,
to quantify the difference.


In [ ]:

from dataclasses import dataclass
from collections import defaultdict, deque

def bond_order_rdkit(b: Chem.Bond) -> float:
    """Return 1,2,3,1.5 for aromatic."""
    bt = b.GetBondType()
    if bt == Chem.rdchem.BondType.SINGLE:
        return 1.0
    if bt == Chem.rdchem.BondType.DOUBLE:
        return 2.0
    if bt == Chem.rdchem.BondType.TRIPLE:
        return 3.0
    if bt == Chem.rdchem.BondType.AROMATIC:
        return 1.5
    # fallback
    return float(b.GetBondTypeAsDouble())

def atoms_by_map_id(mols: List[Chem.Mol]) -> Dict[int, Chem.Atom]:
    out: Dict[int, Chem.Atom] = {}
    for m in mols:
        if m is None:
            continue
        for a in m.GetAtoms():
            mid = int(a.GetAtomMapNum() or 0)
            if mid > 0:
                out[mid] = a
    return out

def bonds_by_map_pair(mols: List[Chem.Mol]) -> Dict[Tuple[int,int], float]:
    out: Dict[Tuple[int,int], float] = {}
    for m in mols:
        if m is None:
            continue
        for b in m.GetBonds():
            ai = int(b.GetBeginAtom().GetAtomMapNum() or 0)
            aj = int(b.GetEndAtom().GetAtomMapNum() or 0)
            if ai <= 0 or aj <= 0:
                continue
            u, v = (ai, aj) if ai < aj else (aj, ai)
            out[(u, v)] = bond_order_rdkit(b)
    return out

def mapped_rxn_to_its(mapped_rxn: str) -> nx.Graph:
    """Mapped reaction SMILES -> ITS graph with r/p bond orders."""
    r_smis, _agents, p_smis = split_rxn_smiles(mapped_rxn)
    r_mols = [Chem.MolFromSmiles(s) for s in r_smis]
    p_mols = [Chem.MolFromSmiles(s) for s in p_smis]

    r_atoms = atoms_by_map_id(r_mols)
    p_atoms = atoms_by_map_id(p_mols)
    V = set(r_atoms) | set(p_atoms)

    G = nx.Graph()
    for mid in sorted(V):
        ar = r_atoms.get(mid)
        ap = p_atoms.get(mid)
        # use available side to set symbol/aromatic; record charges on both sides
        a_any = ar if ar is not None else ap
        if a_any is None:
            continue
        G.add_node(
            mid,
            symbol=a_any.GetSymbol(),
            aromatic=bool(a_any.GetIsAromatic()),
            r_charge=(int(ar.GetFormalCharge()) if ar is not None else None),
            p_charge=(int(ap.GetFormalCharge()) if ap is not None else None),
        )

    r_bonds = bonds_by_map_pair(r_mols)
    p_bonds = bonds_by_map_pair(p_mols)
    for u, v in sorted(set(r_bonds) | set(p_bonds)):
        ro = r_bonds.get((u, v))
        po = p_bonds.get((u, v))
        G.add_edge(u, v, r_order=ro, p_order=po, changed=(ro != po))
    return G

def reaction_center_core_nodes(G: nx.Graph, include_charge_changes: bool = True) -> set[int]:
    C: set[int] = set()
    for u, v, d in G.edges(data=True):
        if d.get("r_order") != d.get("p_order"):
            C.add(u); C.add(v)
    if include_charge_changes:
        for v, nd in G.nodes(data=True):
            rc, pc = nd.get("r_charge"), nd.get("p_charge")
            if (rc is not None) and (pc is not None) and (rc != pc):
                C.add(v)
    return C

def expand_radius(G: nx.Graph, core: set[int], radius: int) -> set[int]:
    if radius <= 0:
        return set(core)
    seen = set(core)
    q = deque([(v, 0) for v in core])
    while q:
        v, d = q.popleft()
        if d >= radius:
            continue
        for u in G.neighbors(v):
            if u not in seen:
                seen.add(u)
                q.append((u, d + 1))
    return seen

@dataclass(frozen=True)
class DPORule:
    L: nx.Graph
    K: nx.Graph
    R: nx.Graph
    meta: Dict[str, object]

def its_to_dpo_rule(G: nx.Graph, core_radius: int = 1, include_charge_changes: bool = True) -> DPORule:
    core = reaction_center_core_nodes(G, include_charge_changes=include_charge_changes)
    C = expand_radius(G, core, radius=core_radius)

    # existence on each side
    L_nodes = [v for v in C if G.nodes[v].get("r_charge") is not None]
    R_nodes = [v for v in C if G.nodes[v].get("p_charge") is not None]
    K_nodes = sorted(set(L_nodes) & set(R_nodes))

    L = nx.Graph()
    R = nx.Graph()
    K = nx.Graph()

    for v in sorted(set(L_nodes) | set(R_nodes)):
        nd = G.nodes[v]
        base = dict(symbol=nd["symbol"], aromatic=nd["aromatic"])
        if v in L_nodes:
            L.add_node(v, **base, formal_charge=int(nd["r_charge"] or 0))
        if v in R_nodes:
            R.add_node(v, **base, formal_charge=int(nd["p_charge"] or 0))
        if v in K_nodes:
            # keep reactant-side charge in K (choice is arbitrary but consistent)
            K.add_node(v, **base, formal_charge=int(nd["r_charge"] or 0))

    for u, v, d in G.edges(data=True):
        ro, po = d.get("r_order"), d.get("p_order")
        if (ro is not None) and (u in L) and (v in L):
            L.add_edge(u, v, order=float(ro))
        if (po is not None) and (u in R) and (v in R):
            R.add_edge(u, v, order=float(po))
        if (ro is not None) and (po is not None) and (ro == po) and (u in K) and (v in K):
            K.add_edge(u, v, order=float(ro))

    meta = {
        "core_nodes": sorted(core),
        "context_radius": int(core_radius),
        "n_L": L.number_of_nodes(),
        "n_R": R.number_of_nodes(),
    }
    return DPORule(L=L, K=K, R=R, meta=meta)

def wl_hash(G: nx.Graph, node_attrs: Sequence[str], edge_attr: str) -> str:
    """
    WL hash that is stable for our educational purposes.
    We stringify attributes to avoid float quirks.
    """
    H = nx.Graph()
    for n, d in G.nodes(data=True):
        H.add_node(n, label="|".join(str(d.get(k)) for k in node_attrs))
    for u, v, d in G.edges(data=True):
        H.add_edge(u, v, elabel=str(d.get(edge_attr)))
    return nx.weisfeiler_lehman_graph_hash(H, node_attr="label", edge_attr="elabel")


In [ ]:

def rule_signature(rule: DPORule) -> Tuple[str, str, str]:
    """Return (hash(L), hash(K), hash(R)) as a lightweight signature."""
    hL = wl_hash(rule.L, node_attrs=("symbol","formal_charge","aromatic"), edge_attr="order")
    hK = wl_hash(rule.K, node_attrs=("symbol","formal_charge","aromatic"), edge_attr="order")
    hR = wl_hash(rule.R, node_attrs=("symbol","formal_charge","aromatic"), edge_attr="order")
    return hL, hK, hR

# --- Compute signatures before/after canonicalization ---
N = min(500, len(df))
rows = []
for rxn in df["mapped_rxn"].dropna().iloc[:N]:
    try:
        its_raw = mapped_rxn_to_its(rxn)
        sig_raw = rule_signature(its_to_dpo_rule(its_raw, core_radius=1))

        rxn_can = canonicalize_mapped_rxn(rxn)
        its_can = mapped_rxn_to_its(rxn_can)
        sig_can = rule_signature(its_to_dpo_rule(its_can, core_radius=1))

        rows.append({"sig_raw": sig_raw, "sig_can": sig_can, "same?": sig_raw == sig_can})
    except Exception:
        continue

res = pd.DataFrame(rows)
print("Evaluated reactions:", len(res))
print("Rule signatures identical (raw vs canonicalized):", res["same?"].mean() if len(res) else None)

if len(res):
    print("\nUnique signatures (raw):", res["sig_raw"].nunique())
    print("Unique signatures (canonicalized):", res["sig_can"].nunique())


## 4. Discussion

If canonicalization works as intended, then:

- Many reactions that only differ by *incidental map numbering* collapse to the same rule signature.
- The fraction `same?` (raw vs canonicalized) may be less than 1 because **raw datasets can contain**
  differences beyond simple renumbering (e.g., different mapping conventions, incomplete mapping, or agents handling).
  Canonicalization does **not** fix those—by design.

In S06–S08 we will assume rules were extracted from **canonicalized mapped reactions** to get:
- stable identifiers for rules,
- stable clustering of reaction centers,
- comparable evaluation metrics.


## 5. Exercises

1. **Alternative tie-breaker.** Modify `canonical_map_renaming` to use product-side ranks first.  
   Compare how many signatures change.
2. **Radius sensitivity.** Repeat the mini-study for `core_radius=0` and `core_radius=2`.  
   Does canonicalization reduce duplicates more at small or large context?
3. **Sanity test.** Pick a reaction, permute map ids with different seeds, and verify canonicalization collapses them.
